Дембіцький Микола Віталійович

### ***DinoV2 + SuperPoint + LightGlue*** UAV-VisLoc

Основні аспекти архітетури - Dinov2 + FAISS для тейлінгу супутникової мапи (ембеддер), SuperPoint + LightGlue для екстракції keypoints та метічгу. Оскільки аналогічний підхід був розглянутий на лекціях, задумка полягала в тому щоб використати його як каркас для іншого backbone - EfficientLoFTR, але через неправильний тайм менеджент (недивлячись на те що за останні 2 дні було витрачено 24 години), проект НЕ реалізований а ні в повному обсязі, а ні на етапі SuperPoint + LightGlue, тому що код містить ряд критичних проблем, які не вдалося виправити до кінця дедлайну.


Тим не менш, пайплайн у сирому вигляді було побудовано з використанням всіх відповідних інструментів. Тобто код - робочий, але не оптимізований ВЗАГАЛІ.


Час виконання не вказано, по причині того що він невідомий, тому що у процесі написання коду використовувався GPU A100 у Colab (підписка була куплена спеціально для цього проету), адже на Т4 пайплайн займав надто багато часу.


У будь-якому випадку, зусилля витрачені на проект були не марні, але всеодно надіюся якщо мої зусилля якимось магічним чином (наприклад не вистачає людей) дозволять приєднатися до тих, хто виконав проект вцілому, тому всеодно надіслав цей notebook на оцінку.

# Preparations

In [ ]:
import os
import zipfile
import gdown

# 1. Вкажіть ID вашого zip-файлу з посилання Google Drive
FILE_ID = "1a2b3c4d5e6f7g8h9i0j"  # Замініть на свій File ID

# 2. Шляхи у внутрішній пам'яті Colab
zip_path = "/content/dataset.zip"
extract_path = "/content"
# https://drive.google.com/file/d/1cmoDAiQhXw1ot7Hm0isJP18zk20iGIYf/view?usp=sharing
# 3. Завантаження через gdown
url = f"https://drive.google.com/uc?id=1cmoDAiQhXw1ot7Hm0isJP18zk20iGIYf"
print("📥 Завантаження zip-архіву...")
gdown.download(url, zip_path, quiet=False)

# 4. Швидке розпакування
print("📦 Розпакування архіву...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

# 5. Видаляємо zip-файл, щоб не займати місце на SSD
if os.path.exists(zip_path):
    os.remove(zip_path)

print("✅ Готово! Файли успішно розміщено в локальній пам'яті.")

📥 Завантаження zip-архіву...


Downloading...
From (original): https://drive.google.com/uc?id=1cmoDAiQhXw1ot7Hm0isJP18zk20iGIYf
From (redirected): https://drive.google.com/uc?id=1cmoDAiQhXw1ot7Hm0isJP18zk20iGIYf&confirm=t&uuid=c3681cac-d76d-44ae-b70d-c317a9fc0dcd
To: /content/dataset.zip
100%|██████████| 2.19G/2.19G [00:18<00:00, 119MB/s] 


📦 Розпакування архіву...
✅ Готово! Файли успішно розміщено в локальній пам'яті.


In [ ]:
%pip install faiss-gpu

!git clone --quiet https://github.com/cvg/LightGlue.git
%cd LightGlue

!pip install -r requirements.txt

!pip install -e .
%cd ..

/content/LightGlue
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 134.5 MB/s eta 0:00:00
Obtaining file:///content/LightGlue
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for lightglue (pyproject.toml) ... done
  Created wheel for lightglue: filename=lightglue-0.0-0.editable-py3-none-any.whl size=14946 sha256=993863f63b95d2619bad58fd28f988171ec15117d92cc3d1afb980c3ed636a95
  Stored in directory: /tmp/pip-ephem-wheel-cache-vhfc_gjt/wheels/cf/9b/1a/81fa7a2b7fcca8ea5e4260afdba28e00ee2453acddfbc8740a
Successfully built lightglue
/content


# Initialzation of functions

In [ ]:
from omegaconf import OmegaConf
from pathlib import Path

base_path = Path("/content")
config = {
    "model": {
        "top_k": 10,
        "num_keypoints": 2048,
    },
    "path": {
        "metadata": str(base_path / "metadata"),
        "faiss": str(base_path / "metadata"),
        "images_dir": str(base_path / "data/example/03/drone"),
        "satellite": str(base_path / "data/example/03/satellite03.tif"),
        "meta": str(base_path / "data/example/03/03.csv"),
    },
    "dino" : {
        "batch_size" : 64,
        "stride" : 518,
        "tile_size" : 1022,
        "embed_dim" : 768,
    },
    "dataset" : {
        "batch_size" : 1,
        "shuffle" : False,
        "val_split" : None,
        "num_workers" : 2,
    },
    "seed" : 147
}
config = OmegaConf.create(config)

In [ ]:
print(config.path.metadata)

/content/metadata


In [ ]:
import cv2
import csv
import torch
import pyproj
import folium
import pandas as pd
import glob
import os
import gc
import numpy as np
import pickle
import faiss
import random
import rasterio
import torch.nn.functional as F
import torchvision.transforms as T
from tqdm import tqdm

from pathlib import Path
from matplotlib import pyplot as plt
from torch.utils.data import Dataset, DataLoader, random_split
from rasterio.windows import Window
from lightglue import SuperPoint, LightGlue
from lightglue.utils import numpy_image_to_torch, rbd


ModuleNotFoundError: No module named 'faiss'

Тут зрозуміло...

In [ ]:
class UAVDataset(Dataset):
    def __init__(self, images_dir, satellite, metadata, transform=None):
        super().__init__()

        self.images_dir = images_dir
        self.satellite = cv2.imread(satellite)
        self.metadata = pd.read_csv(metadata)
        self.transform = transform

        images_path = glob.glob(os.path.join(images_dir, "*.JPG"))

        self.images = sorted(os.path.basename(p) for p in images_path)

    def __getitem__(self, idx):
        image_path = os.path.join(self.images_dir, self.images[idx])
        image_bgr = cv2.imread(image_path)

        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

        meta = self.metadata.iloc[idx]

        if self.transform is not None:
            image_bgr = self.transform(image_bgr)

        return image_rgb, {"lat" : meta['lat'], 'lon' : meta['lon']}, image_bgr

    def __len__(self):
        return len(self.images)

    def get_random_image(self):
        if len(self.images) == 0:
            raise ValueError("Cannot select an image because the dataset is empty.")

        idx = random.randrange(len(self.images))
        return self[idx]

    def get_random_image_data(self):
        if len(self.images) == 0:
            raise ValueError("Cannot select an image because the dataset is empty.")

        idx = random.randrange(len(self.images))

        image_rgb, meta, _ = self[idx]
        image_path = os.path.join(self.images_dir, self.images[idx])
        return {
            "idx" : idx,
            "image_rgb" : image_rgb,
            "path" : image_path,
            "meta" : meta,
        }
    def find_by_name(self, filename: str):
        """
        Пошук фото за назвою (без шляху).
        Повертає дані так само, як __getitem__.
        """
        if filename not in self.images:
            raise ValueError(f"Image '{filename}' not found in dataset.")
        idx = self.images.index(filename)
        return self[idx]


In [ ]:
def build_dataloaders(dataset: UAVDataset, test=True, transform=None):

    common = dict(
        batch_size=config.dataset.batch_size,
        shuffle=config.dataset.shuffle,
        num_workers=config.dataset.num_workers,
        pin_memory=torch.cuda.is_available(),
    )
    if test is not None:
        return DataLoader(dataset, **common)
    return None


In [ ]:
def show_images(number: int = 4, dataset=None, random=True, images_data=None):
    if images_data is not None:
        number = len(images_data)
    cols = 2
    rows = (number + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 3))

    for i, ax in enumerate(axes.flat):
        if i < number:

            if images_data is not None:
                img, meta = images_data['image'], images_data['meta']
                print(img, meta)

            else:
                img, meta = dataset.get_random_image() if random else dataset[i]
            if isinstance(img, torch.Tensor):
                img = img.permute(1, 2, 0).numpy()

            ax.imshow(img)
            try:
                lat = meta["lat"]
                lon = meta["lon"]
                ax.set_title(f"UAV Image {i+1} - Lat: {lat:.6f}, Lon: {lon:.6f}", fontsize=10)
            except KeyError:(
                ax.set_title(f"UAV Image {i+1}", fontsize=10))

        ax.axis("off")

    plt.tight_layout()
    plt.show()

# show_images(4)

Завантаження моделі DinoV2 для тейлінгу

In [ ]:
def load_dinov2(device):
    model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
    model.eval()
    model = model.to(device)
    return model

Функція для обробки батчу вікон (rasterio.window.Window) із застосуванням GeM pooling, для нормалізації даних.

In [ ]:
def _process_batch(model, device, batch_tensors, batch_meta_temp, descriptors_list, metadata_list):
    input_tensor = torch.stack(batch_tensors).to(device)

    with torch.no_grad():
        ret = model.forward_features(input_tensor)
        patches = ret['x_norm_patchtokens']

        patches = patches.clamp(min=1e-6)
        gem_pool = (patches.pow(3).mean(dim=1)).pow(1./3)

        descriptors = F.normalize(gem_pool, p=2, dim=1)

    descriptors_list.extend(descriptors.cpu().numpy())
    metadata_list.extend(batch_meta_temp)

Отримання description для вікна зображенння

In [ ]:
def _extract_description(image_rgb, model, transform=None):
    device = next(model.parameters()).device
    tensor = transform(image_rgb).unsqueeze(0).to(device)
    with torch.no_grad():
        ret = model.forward_features(tensor)
        patches = ret['x_norm_patchtokens']

    patches = patches.clamp(min=1e-6)
    gem_pool = (patches.pow(3).mean(dim=1)).pow(1./3)

    descriptor = F.normalize(gem_pool, p=2, dim=1)
    return descriptor.cpu().numpy()


Основна функція для побудови тейлів:
Відбувається розбиття мапи на вікна, із екстракцією faiss-векторів, для швидкого знаходження схожих вікон серед усіх, та metadata, яка містить інформацію про розташування вікна на мапі.
Результати зберігаються в окремі файли, які створюються автоматично...

In [ ]:
def build_tails_database(model, map_path, transform=None, test=False, batch_size=32):
    if not os.path.exists(os.path.join(config.path.faiss, "indexes.faiss")):
        raise FileExistsError("file indexes.faiss not found")
    if not os.path.exists(os.path.join(config.path.metadata, "metadata.pkl")):
        raise FileExistsError("file metadata.pkl not found")

    print(f"Extracting {map_path} to RAM...")

    with rasterio.open(map_path) as dataset:
        width = dataset.width
        height = dataset.height
        map_transform = dataset.transform

        full_map = dataset.read((1, 2, 3))
        full_map = np.moveaxis(full_map, 0, -1)

    print(f"ap {width}x{height}")

    tile_size = config.dino.tile_size
    stride = config.dino.stride

    descriptors_list = []
    metadata_list = []
    tile_id = 0

    device = next(model.parameters()).device

    batch_tensors = []
    batch_meta_temp = []

    y_range = range(0, height if not test else 1036, stride)
    x_range = range(0, width if not test else 1036, stride)
    total_iterations = len(y_range) * len(x_range)

    with tqdm(total=total_iterations, desc="Загальний прогрес:") as pbar:
        for y in y_range:
            for x in x_range:
                w = min(tile_size, width - x)
                h = min(tile_size, height - y)

                if w < tile_size or h < tile_size:
                    pbar.update(1)
                    continue

                image = full_map[y:y+h, x:x+w]

                center_x = x + (w // 2)
                center_y = y + (h // 2)
                lon, lat = rasterio.transform.xy(map_transform, center_y, center_x)

                if transform is not None:
                    tensor = transform(image)
                    batch_tensors.append(tensor)

                batch_meta_temp.append({
                    "id": tile_id,
                    "window": {"x": x, "y": y, "w": w, "h": h},
                    "lat": lat,
                    "lon": lon
                })

                tile_id += 1
                pbar.update(1)

                if len(batch_tensors) == batch_size:
                    _process_batch(model, device, batch_tensors, batch_meta_temp, descriptors_list, metadata_list)
                    batch_tensors = []
                    batch_meta_temp = []

        if len(batch_tensors) > 0:
            _process_batch(model, device, batch_tensors, batch_meta_temp, descriptors_list, metadata_list)

    del full_map

    description_matrix = np.vstack(descriptors_list).astype("float32")

    faiss_index = faiss.IndexFlatIP(config.dino.embed_dim)
    faiss_index.add(description_matrix)

    faiss.write_index(faiss_index, os.path.join(config.path.faiss, "indexes.faiss"))
    with open(os.path.join(config.path.metadata, "metadata.pkl"), "wb") as f:
        pickle.dump(metadata_list, f)

    print("🎉 Process done!")

Функція виклику білдера (вище).

In [ ]:
def prepare_satellite_map(map_path, model, test=False):
    transform = T.Compose([
        T.ToTensor(),
        T.Resize((config.dino.tile_size, config.dino.tile_size)),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    build_tails_database(model=model, map_path=map_path, transform=transform, test=test, batch_size=config.dino.batch_size)

Отримання найкращих top_k ембеддінгів для заданого фото з датасету.

In [ ]:
def get_top_embeddings(image_rgb, model, top_k=10):
    """
    Input:
        image: RGB - image for extracting embds
        model: Model - moled for embeddings
        top_k: int - number of embeddings (tiles) required
    Output:
        image_embeddings
        tiles_embedding, sorted by score
    """
    faiss_index = faiss.read_index(os.path.join(config.path.faiss, "indexes.faiss"))
    transform = T.Compose([
        T.ToTensor(),
        T.Resize((config.dino.tile_size, config.dino.tile_size)),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    with open(os.path.join(config.path.metadata, "metadata.pkl"), "rb") as f:
        metadata = pickle.load(f)

    image_embd = _extract_description(image_rgb=image_rgb, model=model, transform=transform)

    distances, indices = faiss_index.search(image_embd, top_k)

    tiles = []
    with rasterio.open(config.path.satellite) as dataset:
        for i in range(top_k):
            idx = indices[0][i]
            score = distances[0][i]
            meta = metadata[idx]
            win = meta["window"]

            window = Window(col_off=win["x"], row_off=win["y"], width=win["w"], height=win["h"])
            tile_array = dataset.read((1, 2, 3), window=window)
            tile_rgb_cv2 = np.moveaxis(tile_array, 0, -1)

            tiles.append({
                "tile_id": idx,
                "score": float(score),
                "image": tile_rgb_cv2,
                "metadata" : meta,
            })

    return image_embd, sorted(tiles, key=lambda x: x["score"])


Модель для застосування SuperPoint + LightGlue

In [ ]:
class Model():
    """SuperPoint + LightGlue model."""
    def __init__(self, device=None):
        self.device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu") if device is None else device
        self.extractor = SuperPoint(num_keypoints=config.model.num_keypoints).eval().to(self.device)
        self.matcher = LightGlue(features="superpoint").eval().to(self.device)

    def __call__(self, image, tiles, top_k=config.model.top_k):
        images_kpts = self._get_keypoints(image)
        tiles_kpts = [self._get_keypoints(tile_kpts) for tile_kpts in tiles_kpts]

        matches = [self._get_matches(images_kpts, tile_kpts) for tile_kpts in tiles_kpts]

        images_kpts, tiles_kpts = self._rbd(images_kpts, tiles_kpts)

        images_kpts_matched, tiles_kpts_matched = [], []
        for tile_kpts, mat, tile in zip(tiles_kpts, matches, tiles):
            tile_kpts_matched = tile_kpts[mat[..., 1]]

            win = tile['metadata']['window']

            x0, y0 = win['x'], win['y']

            offset = torch.tensor([x0, y0], device=tile_kpts_matched.device, dtype=tile_kpts_matched.dtype)

            images_kpts_matched.append(images_kpts[mat[..., 0]])
            tiles_kpts_matched.append({'keypoints' : tile_kpts_matched, 'offset' : offset})

        return images_kpts_matched, tiles_kpts_matched

    def _rbd(image_kpts, tiles_kpts):
        return rbd(image_kpts), [rbd(tile_kpts) for tile_kpts in tiles_kpts]

    def _get_keypoints(self, image):
        if isinstance(image, np.array):
            image = numpy_image_to_torch(image).to(self.device)
        with torch.no_grad():
            keypoints = self.extractor.extract(image)['keypoints']
        return keypoints

    def _get_matches(self, image_kpts, tile_kpts):
        matches01 = self.matcher({"image0": image_kpts, 'image1': tile_kpts})
        return rbd(matches01)['matches']


Обчислення гомографії із застосуванням RANSAC

In [ ]:
def _get_ransac_and_homography(kpts_drone, kpts_sat, device, reproj_threshold=5):
    """
    Calculating homography for keypoints after processing RANSAC...
    Input:
        kpts_drone - image (drone) keypoints
        kpts_sat - satellite map keypoints
        device - torch.device
        reproj_threshold=5 - threshold
    Output:
        Homography, inliers_count, mask
    """
    if isinstance(kpts_drone, torch.Tensor):
        pts_drone = np.ascontiguousarray(kpts_drone.detach().cpu().numpy(), dtype=np.float32)
    else:
        pts_drone = np.ascontiguousarray(kpts_drone, dtype=np.float32)

    if isinstance(kpts_sat, torch.Tensor):
        pts_sat = np.ascontiguousarray(kpts_sat.detach().cpu().numpy(), dtype=np.float32)
    else:
        pts_sat = np.ascontiguousarray(kpts_sat, dtype=np.float32)

    print(f"pts_drone: {pts_drone.shape}, pts_sat: {pts_sat.shape}")

    if pts_drone.shape[0] < 4 or pts_sat.shape[0] < 4:
        return None, 0, None

    H, mask = cv2.findHomography(pts_drone, pts_sat, cv2.RANSAC, reproj_threshold)
    inliers_count = int(mask.sum()) if mask is not None else 0

    return H, inliers_count, mask


Функція знаходить найкращий з валідних тейлів, тобто той, що має найбільше метчів, та підходить для побудови гомографії

In [ ]:
def find_valid_tile_batched(image_rgb, tiles_feats, sp_lg_model, device, min_inliers=15, chunk_size=30):
    """
    Function searches for best possible tile for some image.
    Inputs:
        image (RGB) - for extracting h, w
        tiles_faets - tiles info
    """
    h_drone, w_drone = image_rgb.shape[:2]
    center_pixel_drone = np.array([[[h_drone // 2, w_drone // 2]]], dtype=np.float32)

    for i in range(0, len(tiles_feats), chunk_size):
        chunk_tiles_feats = tiles_feats[i : i + chunk_size]

        image_kpts_list, tiles_kpts_list = sp_lg_model(image_rgb, chunk_tiles_feats)

        best_homography = None
        best_inliers = min_inliers - 1
        best_idx = -1
        best_target = None
        best_drone_kpts = None
        best_tile_kpts_data = None

        for j, (image_kpts, tile_kpts_data) in enumerate(zip(image_kpts_list, tiles_kpts_list)):
            _, inliers_count, _ = _get_ransac_and_homography(
                image_kpts, tile_kpts_data['keypoints'], device, reproj_threshold=5
            )

            if inliers_count > best_inliers and homography is not None:
                best_inliers = inliers_count
                best_idx = i + j
                best_drone_kpts = image_kpts
                best_tile_kpts_data = tile_kpts_data

        torch.cuda.empty_cache()
        gc.collect()

        if best_homography is None:
            return None, None, None

        return best_drone_kpts, best_tile_kpts_data, {"id": best_idx, "inliers": best_inliers}



Повертає RASTERIO_MAP.transform

In [ ]:
def _get_map_transform(file=config.path.satellite):
    """
    Returns satellite map's transform.
    Input:
        file - path to sat. map
    Output:
        transform
    """
    with rasterio.open(file) as sat_im:
        bounds = sat_im.bounds
        MAP_TL_LON, MAP_TL_LAT = bounds.left, bounds.top
        MAP_BR_LON, MAP_BR_LAT = bounds.right, bounds.bottom

        print(f"Satelite map bounds:")
        print(f"Top-Left: Lat {MAP_TL_LAT:.6f}, Lon {MAP_TL_LON:.6f}")
        print(f"Bottom-Right: Lat {MAP_BR_LAT:.6f}, Lon {MAP_BR_LON:.6f}")

        transform = sat_im.transform
        return transform


Функція для визначення геолокації (якщо можливо)

In [ ]:
def calculate_geoposition(image, drone_kpts, sat_kpts, offset):
    if isinstance(drone_kpts, torch.Tensor):
        drone_kpts = drone_kpts.detach().cpu().numpy()
    if isinstance(sat_kpts, torch.Tensor):
        sat_kpts = sat_kpts.detach().cpu().numpy()
    if isinstance(offset, torch.Tensor):
        offset = offset.detach().cpu().numpy()

    global_sat_kpts = sat_kpts + offset

    object_points = np.zeros((len(global_sat_kpts), 3), dtype=np.float32)
    object_points[:, :2] = global_sat_kpts

    tensor_kpts = torch.tensor(drone_kpts, dtype=torch.float32)

    image_points = tensor_kpts.detach().cpu().numpy()

    focal_length = 500
    cx, cy = image.shape[2]/2, image.shape[1]/2
    K = np.array([[focal_length, 0, cx],
                  [0, focal_length, cy],
                  [0, 0, 1]], dtype=np.float32)
    dist_coeffs = np.zeros((4, 1))

    success, rvec, tvec, inliers = cv2.solvePnPRansac(
        object_points, image_points, K, dist_coeffs
    )

    if success:
        px_x, px_y = int(tvec[0][0]), int(tvec[1][0])
        map_transform = _get_map_transform()
        lon, lat = rasterio.transform.xy(map_transform, px_y, px_x)
        return lon, lat
    else:
        raise ValueError("(RANSAC Error) Camera position could not be calculated.")


MSE похибка з тестового завдання

In [ ]:
def mean_haversine_error(y_true, y_pred, R=6371000):
    """
    Calculates MSE metric.
    """

    lat_true = np.radians(y_true[:, 0])
    lon_true = np.radians(y_true[:, 1])
    lat_pred = np.radians(y_pred[:, 0])
    lon_pred = np.radians(y_pred[:, 1])

    dlat = lat_pred - lat_true
    dlon = lon_pred - lon_true

    a = np.sin(dlat / 2) ** 2 + np.cos(lat_true) * np.cos(lat_pred) * np.sin(dlon / 2) ** 2
    d = 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

    return np.mean(d)

Автоматичне створення необхідних папок і файлів

In [ ]:
def setup_dirs_and_files(clear_if_exists: bool = False):
    """
    Створює папки та файли, якщо вони відсутні.
    Якщо файли вже існують і clear_if_exists=True, вони повністю очищаються.
    """
    # Словник з файлами та відповідними папками
    targets = {
        Path(config.path.faiss): "indexes.faiss",
        Path(config.path.metadata): "metadata.pkl"
    }

    for dir_path, filename in targets.items():
        # 1. Створюємо папку (якщо її немає)
        dir_path.mkdir(parents=True, exist_ok=True)

        file_path = dir_path / filename

        # 2. Якщо файл існує і потрібна очистка
        if file_path.exists() and clear_if_exists:
            file_path.write_bytes(b"")  # Очищаємо вміст файлу

        # 3. Якщо файлу немає — створюємо порожній файл
        elif not file_path.exists():
            file_path.touch()

Отримання центру карти (альтернативні координати для пердікшина)

In [ ]:
def get_map_center(file_path=config.path.satellite):
    """
    Повертає географічні координати (довготу та широту) центру супутникової мапи.
    Використовується як резервне значення (fallback) при повній втраті локалізації.
    """
    with rasterio.open(file_path) as dataset:
        bounds = dataset.bounds
        center_lon = (bounds.left + bounds.right) / 2.0
        center_lat = (bounds.top + bounds.bottom) / 2.0

    return center_lon, center_lat

# Main

In [ ]:
def main():
    setup_dirs_and_files(clear_if_exists=True)
    device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

    print("Downloading DINOv2 & SuperPoint+LightGlue...")
    dinov2_model = load_dinov2(device)
    sp_lg_model = Model(device=device)

    random.seed(config.seed)
    dataset = UAVDataset(
        config.path.images_dir, config.path.satellite, config.path.meta
    )

    loader = build_dataloaders(dataset)
    print(f"N batches: {len(loader)}")

    prepare_satellite_map(map_path=config.path.satellite, model=dinov2_model, test=False)

    fallback_lon, fallback_lat = get_map_center(config.path.satellite)
    output_csv = "predictions.csv"

    with open(output_csv, mode='w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['filename', 'gt_lat', 'gt_lon', 'pred_lat', 'pred_lon', 'error_m', 'top1_similarity'])

        for batch_idx, (image_rgb_batch, meta_batch, _) in enumerate(loader):

            image_rgb = image_rgb_batch[0].cpu().numpy()

            filename = dataset.images[batch_idx]

            try:
                gt_lat = float(meta_batch['lat'][0])
                gt_lon = float(meta_batch['lon'][0])
            except (KeyError, TypeError):
                gt_lat, gt_lon = 0.0, 0.0

            emb_drone, tiles_feats_10 = get_top_embeddings(image_rgb=image_rgb, model=dinov2_model, top_k=10)

            top1_similarity = tiles_feats_10[0]['score'] if len(tiles_feats_10) > 0 else 0.0

            best_dkpts, best_tkpts, best_info = find_valid_tile_batched(
                image_rgb, tiles_feats_10, sp_lg_model, device, min_inliers=15, chunk_size=10
            )

            if best_dkpts is None:
                only_way = tiles_feats_10[0]['metadata']['window']
            else:
                try:
                    pred_lon, pred_lat = calculate_geoposition(image_rgb, best_dkpts, best_tkpts)
                except ValueError as e:
                    pred_lon, pred_lat = fallback_lon, fallback_lat

            y_true = np.array([[gt_lat, gt_lon]])
            y_pred = np.array([[pred_lat, pred_lon]])
            error_m = mean_haversine_error(y_true, y_pred)

            writer.writerow([filename, gt_lat, gt_lon, pred_lat, pred_lon, error_m, top1_similarity])
            f.flush()


main()

Код, який використовувався для дебагінгу...
Залишив для proof of work

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# # Збереження файлу у вашу папку Drive
# import shutil
# shutil.copy("metadata/metadata.pkl", "/content/drive/MyDrive/metadata.pkl")
# shutil.copy("faiss/indexes.faiss", "/content/drive/MyDrive/indixes.faiss")

In [ ]:
# setup_dirs_and_files(clear_if_exists=False)
# device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

# dinov2_model = load_dinov2(device)

# random.seed(config.seed)
# dataset = UAVDataset(
#     config.path.images_dir, config.path.satellite, config.path.meta
# )
# loader = build_dataloaders(dataset)
# print(f"Dataset contains {len(loader)} batches.")

# #

# model = Model()

# # pbar = tqdm(loader, desc="Processing images:")
# # for image_rgb, meta, image_bgr in pbar:
#     # image_rgb = image_rgb.to(device, non_blocking=True)
#     # meta = meta.to(device, non_blocking=True)
# image_rgb, meta, _ = dataset.find_by_name('03_0010.JPG')
# plt.imshow(image_rgb)

In [ ]:
# embeddings_drone, tiles_feats = get_top_embeddings(image_rgb=image_rgb, model=dinov2_model, top_k=768)

In [ ]:
# meta = [tile['metadata'] for tile in tiles_feats]
# for data in meta:
#     print(f'lat {data['lat']}, lon {data['lon']}')

In [ ]:
# model = Model()
# image_kpts, tiles_kpts, matches = model(image_rgb, tiles_feats)

In [ ]:
# print(meta['lon'])

In [ ]:
# print(meta['lat'])


In [ ]:
# best_drone_kpts, best_tile_kpts, best_tile_info = find_valid_tile_batched(
#     image_rgb, tiles_feats, model, device=device, min_inliers=15
# )

In [ ]:
# Satelite map bounds:
# Top-Left: Lat 32.355491, Lon 119.805926
# Bottom-Right: Lat 32.290290, Lon 119.900052
#  9104.27 м | Координати: [32.353182, 119.807856], gt: [32.30931925, 119.8896768]

In [ ]:
# calculate_geoposition(image_rgb, best_drone_kpts, best_tile_kpts)

In [ ]:
# model = Model()
# MIN_INLIERS_THRESHOLD = 15

# top_k_first = 10
# _, tiles_feats_10 = get_top_embeddings(image_rgb=image_rgb, model=dinov2_model, top_k=top_k_first)

# best_drone_kpts, best_tile_kpts, best_tile_info = find_valid_tile_batched(
#     image_rgb, tiles_feats_10, model, device=device, min_inliers=MIN_INLIERS_THRESHOLD
# )

# if best_drone_kpts is None:
#     top_k_total = 30

#     _, tiles_feats_30 = get_top_embeddings(image_rgb=image_rgb, model=dinov2_model, top_k=top_k_total)

#     new_tiles_feats = tiles_feats_30[top_k_first:]

#     best_drone_kpts, best_tile_kpts, best_tile_info = find_valid_tile_batched(
#         image_rgb, new_tiles_feats, model, device=device, min_inliers=MIN_INLIERS_THRESHOLD
#     )

#     if best_tile_info is not None:
#         best_tile_info["id"] += top_k_first
# if best_drone_kpts is None:

#     lon, lat = get_map_center())

# else:
#     try:
#         lon, lat = calculate_geoposition(image_rgb, best_drone_kpts, best_tile_kpts)
#     except ValueError as e:

#         lon, lat = get_map_center()



In [ ]:
# images = [feat['image'] for feat in tiles_feats]
# print(len(images))

In [ ]:
# import matplotlib.pyplot as plt

# def show_images(images, n=10, rows=3):
#     """
#     Виводить n зображень зі списку images у rows рядів.
#     """
#     n = min(n, len(images))
#     cols = (n + rows - 1) // rows  # кількість колонок

#     plt.figure(figsize=(3 * cols, 3 * rows))

#     for i in range(n):
#         plt.subplot(rows, cols, i + 1)
#         plt.imshow(images[i])
#         plt.axis("off")
#         plt.title(f"Image {i+1}")

#     plt.tight_layout()
#     plt.show()


In [ ]:
# show_images(images)

In [ ]:
# dinov2_model = load_dinov2()
# device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
#
# random.seed(config.seed)
# dataset = UAVDataset(
#     config.path.images_dir, config.path.satellite, config.path.meta
# )
# loader = build_dataloaders(dataset)
# print(f"Dataset contains {len(loader)} batches.")
#
# image_rgb, meta, _ = dataset.get_random_image()
#
# embeddings_drone, tiles_feats = get_top_embeddings(image_rgb=image_rgb, model=dinov2_model, top_k=config.model.top_k)
#
# model = Model()
#
# image_kpts, tiles_kpts, matches = model(embeddings_drone, tiles_feats)
#
# best_tile_kpts, best_tile_info = get_best_tile(image_rgb, image_kpts, tiles_kpts)
#
#
# try:
#     lon, lat = calculate_geoposition(image_rgb, image_kpts, best_tile_kpts)
#     print(f"Lon: {lon:.6f}, Lat: {lat:.6f}")
# except ValueError as e:
#     print(f"Error calculating geoposition: {e}")


In [ ]:
# lon, lat = rasterio.transform.xy(_get_map_transform(), 10000, 10000)

# print(lon, lat)